# 08.8 LoRA 个性化：配套音频与 ACE-Step 训练入口

本 Notebook 展示个性化训练的工程组织：资产检查、manifest、LoRA 配置、训练命令和输出记录。当前没有 `audio_author/chapter_08_author` 时只打印下载说明，不用其他数据集替代。


In [ ]:
from pathlib import Path
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from _common.config import load_yaml_config
from _common.dataset_registry import asset_status
from _common.paths import portable_path
from _common.tables import write_rows
from finetune.ace_step_lora_dataset import load_lora_manifest, validate_lora_manifest
from finetune.ace_step_lora_train import write_lora_train_plan
from finetune.musicgen_finetune import (
    MUSICGEN_ISSUE_FIELDS,
    export_audiocraft_jsonl,
    load_musicgen_manifest,
    validate_musicgen_manifest,
    write_musicgen_finetune_plan,
)
from model_runners.conditioning import build_conditioning_rows

OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

def rel(path):
    return portable_path(path, ROOT)


## 哪些模型能自己训练？

本章区分三种情况：toy 模型从头训练、开源工程的微调/LoRA、以及只适合调用预训练权重的大模型。WaveNet 与 mini Codec-LM 已在 08_3/08_5 从头训练；本 Notebook 只处理真实大模型里的个性化路径。


In [ ]:
trainability = pd.DataFrame(build_conditioning_rows())[
    ["model_id", "trainability_in_chapter", "practical_training_path", "current_demo_conditions"]
]
display(trainability)
prepared_segments = OUTPUT_TABLES / "08_0_prepared_segments.csv"
if prepared_segments.exists():
    prepared_df = pd.read_csv(prepared_segments)
    display(prepared_df.head(10))
else:
    print("Run 08_0_audio_data_pipeline.ipynb first to create cleaned 8-second author-audio segments.")


In [ ]:
asset = asset_status("audio_author_ch08")
asset_row = {
    "asset_id": asset.spec.asset_id,
    "available": asset.ok,
    "paths": ";".join(rel(path) for path in asset.existing_paths),
    "download_hint": asset.spec.download_hint,
}
display(pd.DataFrame([asset_row]))
if not asset.ok:
    print(asset.spec.download_hint)


In [ ]:
example_manifest = ROOT / "data_manifests" / "audio_author_ch08.example.csv"
example_df = pd.read_csv(example_manifest)
display(example_df)
print("Use this schema for data_manifests/audio_author_ch08.csv when adding more author audio.")


In [ ]:
config = load_yaml_config(ROOT / "configs" / "ace_step_lora.yaml")
plan_rows = write_lora_train_plan(
    OUTPUT_TABLES / "08_8_lora_plan.csv",
    config=config,
    config_path="configs/ace_step_lora.yaml",
    chapter_root=ROOT,
)
display(pd.DataFrame(plan_rows))


In [ ]:
musicgen_config = load_yaml_config(ROOT / "configs" / "musicgen_finetune.yaml")
musicgen_plan_rows = write_musicgen_finetune_plan(
    OUTPUT_TABLES / "08_8_musicgen_finetune_plan.csv",
    config=musicgen_config,
    chapter_root=ROOT,
)
display(pd.DataFrame(musicgen_plan_rows))


In [ ]:
manifest_path = ROOT / config["data"]["manifest_csv"]
if asset.ok and manifest_path.exists():
    issues = validate_lora_manifest(
        manifest_path,
        audio_root=ROOT / config["data"]["audio_root"],
        require_files=True,
    )
    write_rows(
        OUTPUT_TABLES / "08_8_lora_manifest_issues.csv",
        [issue.as_row() for issue in issues],
        fieldnames=["row_index", "case_id", "field", "severity", "message"],
    )
    if issues:
        display(pd.DataFrame([issue.as_row() for issue in issues]))
        raise RuntimeError("LoRA manifest validation failed.")
    items = load_lora_manifest(manifest_path, audio_root=ROOT / config["data"]["audio_root"])
    display(pd.DataFrame([{**item.__dict__, "path": rel(item.path)} for item in items]))

    musicgen_issues = validate_musicgen_manifest(
        manifest_path,
        audio_root=ROOT / musicgen_config["data"]["audio_root"],
        require_files=True,
    )
    write_rows(
        OUTPUT_TABLES / "08_8_musicgen_manifest_issues.csv",
        [issue.as_row() for issue in musicgen_issues],
        fieldnames=MUSICGEN_ISSUE_FIELDS,
    )
    if musicgen_issues:
        display(pd.DataFrame([issue.as_row() for issue in musicgen_issues]))
        raise RuntimeError("MusicGen fine-tuning manifest validation failed.")
    musicgen_items = load_musicgen_manifest(
        manifest_path,
        audio_root=ROOT / musicgen_config["data"]["audio_root"],
    )
    export_audiocraft_jsonl(
        musicgen_items,
        ROOT / musicgen_config["data"]["train_jsonl"],
        split="train",
    )
    export_audiocraft_jsonl(
        musicgen_items,
        ROOT / musicgen_config["data"]["valid_jsonl"],
        split="valid",
    )
else:
    print("LoRA training main flow stopped: author asset pack or manifest is not available.")
    print("Expected manifest:", rel(manifest_path))
